Import required packages

In [18]:
import pandas as pd
import numpy as np
import math

Read raw data downloaded from API endpoints and stored in `..\data` directory

In [19]:
ebird_co_df = pd.read_csv("..\\data\\ebird_co.csv")
weather_stations_co_df = pd.read_csv("..\\data\\co_weather_stations.csv")
weather_co_df = pd.read_csv("..\\data\\co_weather.csv")
natural_disasters_co_df = pd.read_csv("..\\data\\co_natural_disasters.csv")
urban_pop_co_df = pd.read_csv("..\\data\\co_urban_pop.csv")

## Data Cleaning

## Data Preprocessing

### Bird sightings data

In [20]:
# 1. Map bird sighting lat-long coordinates to CO counties by finding the county whose centroid is closest to the coordinates in terms of Euclidean distance.

# Function to find nearest county to a set of lat-long co-ordinates by calculating Euclidean distances to county centroids
def get_nearest_county(lat, lon):
    nearest_county = None
    min_dist = float('inf')

    # List of CO county centroid coorindates
    co_county_centroids = {
        "Adams": (39.8743, -104.3318),
        "Alamosa": (37.5741, -105.7866),
        "Arapahoe": (39.6524, -104.6747),
        "Archuleta": (37.1900, -107.0500),
        "Baca": (37.3200, -102.5600),
        "Bent": (37.9550, -103.0720),
        "Boulder": (40.0567, -105.3291),
        "Broomfield": (39.9497, -105.0705),
        "Chaffee": (38.7398, -106.1846),
        "Cheyenne": (38.8280, -102.6030),
        "Clear Creek": (39.7210, -105.6581),
        "Conejos": (37.2010, -106.1920),
        "Costilla": (37.2780, -105.4280),
        "Crowley": (38.3270, -103.7850),
        "Custer": (38.1090, -105.3670),
        "Delta": (38.8610, -107.8630),
        "Denver": (39.7048, -104.9555),
        "Dolores": (37.7476, -108.5304),
        "Douglas": (39.3484, -105.0939),
        "Eagle": (39.6280, -106.6950),
        "El Paso": (38.9268, -104.5696),
        "Elbert": (39.2870, -104.1360),
        "Fremont": (38.4730, -105.4400),
        "Garfield": (39.5990, -107.9040),
        "Gilpin": (39.8594, -105.5166),
        "Grand": (40.1565, -106.0356),
        "Gunnison": (38.6670, -107.0320),
        "Hinsdale": (37.8200, -107.2800),
        "Huerfano": (37.6847, -104.9606),
        "Jackson": (40.6664, -106.3428),
        "Jefferson": (39.5864, -105.2505),
        "Kiowa": (38.4326, -102.7402),
        "Kit Carson": (39.3055, -102.6030),
        "La Plata": (37.2866, -107.8433),
        "Lake": (39.2025, -106.3448),
        "Larimer": (40.6664, -105.4612),
        "Las Animas": (37.3158, -104.0387),
        "Lincoln": (38.9881, -103.5140),
        "Logan": (40.7247, -103.1101),
        "Mesa": (39.0183, -108.4664),
        "Mineral": (37.6689, -106.9241),
        "Moffat": (40.6184, -108.2074),
        "Montezuma": (37.3386, -108.5966),
        "Montrose": (38.4022, -108.2693),
        "Morgan": (40.2626, -103.8097),
        "Otero": (37.9026, -103.7165),
        "Ouray": (38.1555, -107.7693),
        "Park": (39.1194, -105.7172),
        "Phillips": (40.5940, -102.3576),
        "Pitkin": (39.2171, -106.9166),
        "Prowers": (37.9552, -102.3934),
        "Pueblo": (38.1735, -104.5127),
        "Rio Blanco": (39.9798, -108.2171),
        "Rio Grande": (37.5825, -106.3832),
        "Routt": (40.4851, -106.9913),
        "Saguache": (38.0805, -106.2815),
        "San Juan": (37.7640, -107.6762),
        "San Miguel": (38.0038, -108.4057),
        "Sedgwick": (40.8759, -102.3518),
        "Summit": (39.6342, -106.1164),
        "Teller": (38.8824, -105.1617),
        "Washington": (39.9710, -103.2012),
        "Weld": (40.5548, -104.3925),
        "Yuma": (40.0029, -102.4243)
        }

    for county, (olat, olon) in co_county_centroids.items():
        # Simple Euclidean distance (sufficient for "nearest" classification in small areas)
        dist = math.sqrt((lat - olat)**2 + (lon - olon)**2)

        if dist < min_dist:
            min_dist = dist
            nearest_county = county

    return nearest_county

# Create new column to store counties where birds were sighted for each observation
ebird_co_df["county"] = [get_nearest_county(lat, lon) for lat, lon in zip(ebird_co_df['lat'], ebird_co_df['lng'])]

### Weather station data

In [5]:
# To reduce amount of data, select the central weather station for each location
# These stations' names don't have numbers
central_stations_df = weather_stations_co_df[weather_stations_co_df["name"].str.match(r"^[^\d]+$")]

central_stations_df

,Unnamed: 0,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude
350,350,1898.9,2019-10-01,2022-11-01,39.45331,"THE PINERY, CO US",0.8681,GHCND:US1CODG0294,METERS,-104.739092
842,842,1527.0,1998-06-01,2026-01-01,40.57590,"FCL, CO US",0.9848,GHCND:US1COLR0255,METERS,-105.085800
1360,1360,1718.2,1947-07-01,2025-12-01,39.49920,"ALTENBERN, CO US",0.9946,GHCND:USC00050214,METERS,-108.380900
1361,1361,2724.6,1961-06-01,2026-01-01,38.99305,"ANTERO RESERVOIR, CO US",0.9870,GHCND:USC00050263,METERS,-105.891450
1362,1362,1225.3,1948-09-01,2024-09-01,38.85280,"ARAPAHOE, CO US",0.4096,GHCND:USC00050304,METERS,-102.176400
...,...,...,...,...,...,...,...,...,...,...
1755,1755,1779.4,1967-02-01,2026-01-01,38.67833,"FORT CARSON BUTTS ARMY AIR FIELD, CO US",0.3757,GHCND:USW00094015,METERS,-104.756670
1756,1756,2011.7,2017-08-01,2026-01-01,40.48111,"HAYDEN YAMPA VALLEY AIRPORT, CO US",0.9113,GHCND:USW00094025,METERS,-107.217500
1757,1757,1526.7,2011-10-01,2026-01-01,40.45000,"FORT COLLINS LOVELAND AIRPORT, CO US",0.6277,GHCND:USW00094035,METERS,-105.016670
1758,1758,1940.8,1997-06-01,2026-01-01,40.04437,"MEEKER AIRPORT, CO US",0.9911,GHCND:USW00094050,METERS,-107.888360


### Weather data

In [12]:
# Pivot weather_df to wide format such that each data type becomes its own column
weather_co_df = weather_co_df.pivot(
    index = ["date", "station", "attributes"],
    columns = "datatype",
    values = "value"
).reset_index()

# Join GSOM data with stations data
weather_and_stations_df = pd.merge(
    left = weather_co_df,
    right = central_stations_df, 
    how = "left", 
    left_on = "station", 
    right_on = "id", 
    indicator = True
    )

# Post-join stats
# _merge
# left_only     57559
# both          50223
# right_only        0
weather_and_stations_df

,date,station,attributes,AWND,PRCP,RHMN,RHMX,TMAX,TMIN,Unnamed: 0,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude,_merge
0,2021-01-01T00:00:00,GHCND:US1COAD0022,"2,,,N",NaN,7.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,2021-01-01T00:00:00,GHCND:US1COAD0087,"4,,,N",NaN,5.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,2021-01-01T00:00:00,GHCND:US1COAD0100,",,,N",NaN,12.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,2021-01-01T00:00:00,GHCND:US1COAD0120,",,,N",NaN,5.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,2021-01-01T00:00:00,GHCND:US1COAD0123,"1,,,N",NaN,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70679,2025-12-01T00:00:00,GHCND:USW00094050,",,,D",NaN,12.0,NaN,NaN,8.15,-6.45,1758.0,1940.8,1997-06-01,2026-01-01,40.04437,"MEEKER AIRPORT, CO US",0.9911,GHCND:USW00094050,METERS,-107.88836,both
70680,2025-12-01T00:00:00,GHCND:USW00094050,",1",2.3,NaN,NaN,NaN,NaN,NaN,1758.0,1940.8,1997-06-01,2026-01-01,40.04437,"MEEKER AIRPORT, CO US",0.9911,GHCND:USW00094050,METERS,-107.88836,both
70681,2025-12-01T00:00:00,GHCND:USW00094074,",,,R",NaN,2.7,NaN,NaN,11.97,-5.53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
70682,2025-12-01T00:00:00,GHCND:USW00094075,",,,R",NaN,38.6,NaN,NaN,2.45,-4.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [13]:
weather_and_stations_df["_merge"].value_counts()

_merge
left_only     49364
both          21320
right_only        0
Name: count, dtype: int64

### Urban population data

In [ ]:
'''
Use Urban Percentage for analysis. Bin values into 20%-wide buckets.
'''

In [ ]:
# To estimate county populations from 2021 to 2025, assume an annual growth rate of 0.75% (0.075)
# Source: U.S. Census Bureau, Colorado State Demography Office
r = 0.075

# Calculate exponential growth for 2021-2025
# Formula: P(t) = P0 * e^(rt)
for year in range(2021, 2026):
    # Calculate time 't' (years since 2020)
    t = year - 2020

    # Calculate total population, round to nearest whole number, and convert to integer
    column_name = f'Total Pop {year}'
    co_urban_df[column_name] = (co_urban_df['Total Pop'] * np.exp(r * t)).round(0).astype(int)

    # Calculate urban population, round to nearest whole number, and convert to integer
    column_name = f'Urban Pop {year}'
    co_urban_df[column_name] = (co_urban_df['Urban Pop'] * np.exp(r * t)).round(0).astype(int)

    # Calculate urban population percentage, round to 2 decimal places
    column_name = f'Urban Percentage {year}'
    co_urban_df[column_name] = ((co_urban_df[f'Urban Pop {year}'] / co_urban_df[f'Total Pop {year}'])*100).round(2)

# Mark if County has urban areas, 1 if urban pop > 0, 0 otherwise
co_urban_df['Urban County'] = np.where(co_urban_df['Urban Pop'] > 0, 1, 0)

### Natural disaster data

In [15]:
natural_disasters_co_df

,Unnamed: 0,femaDeclarationString,disasterNumber,state,declarationType,declarationDate,fyDeclared,incidentType,declarationTitle,ihProgramDeclared,...,placeCode,designatedArea,declarationRequestNumber,lastIAFilingDate,incidentId,region,designatedIncidentTypes,lastRefresh,hash,id
0,0,FM-5423-CO,5423,CO,FM,2021-12-30T00:00:00.000Z,2022,Fire,MARSHALL FIRE,False,...,99013,Boulder (County),21132,NaN,2021123101,8,NaN,2024-08-27T18:22:14.800Z,3f0e9054a42676d0d7f75b6279f972d18c4b39bb,b8cb17aa-bb52-45b5-80ed-3dcd3e2c7486
1,1,DR-4634-CO,4634,CO,DR,2021-12-31T00:00:00.000Z,2022,Fire,WILDFIRES AND STRAIGHT-LINE WINDS,True,...,99013,Boulder (County),22001,2022-03-02T00:00:00.000Z,2022010101,8,NaN,2024-08-27T18:22:14.800Z,3f2ce967d16d48c7a12b74a1c31859243f1ed5fa,b61bd47d-de3a-4a86-887e-94595817d28f
2,2,DR-4581-CO,4581,CO,DR,2021-01-15T00:00:00.000Z,2021,Fire,WILDFIRES,False,...,99049,Grand (County),20328,NaN,2020122301,8,NaN,2024-08-27T18:22:14.800Z,3f9db442ca63530ecfcf6603b3e4a08401ecabd5,10e97111-a9ad-4270-b86f-8ec936e0aa8c
3,3,DR-4581-CO,4581,CO,DR,2021-01-15T00:00:00.000Z,2021,Fire,WILDFIRES,False,...,99069,Larimer (County),20328,NaN,2020122301,8,NaN,2024-08-27T18:22:14.800Z,ec0a151b3cdbe45d5eb94d018f23ec417af9533e,98e30bb6-82ac-4838-8f4e-c547fdb9006c
4,4,FM-5604-CO,5604,CO,FM,2025-08-06T00:00:00.000Z,2025,Fire,ELK FIRE,False,...,99103,Rio Blanco (County),25109,NaN,2025080701,8,R,2025-08-08T19:42:11.470Z,889b9529a11f0a115d0682b03ce22da55239cfba,b4d54825-9061-437e-89dd-6db50192d3a4
5,5,FM-5603-CO,5603,CO,FM,2025-08-06T00:00:00.000Z,2025,Fire,LEE FIRE,False,...,99103,Rio Blanco (County),25108,NaN,2025080702,8,R,2025-08-08T19:42:11.470Z,73656c8219ea708cd77945c00a931f269eae775f,28b8935e-4883-443a-877e-8fcb6441b389
6,6,FM-5606-CO,5606,CO,FM,2025-08-11T00:00:00.000Z,2025,Fire,OAK FIRE,False,...,99007,Archuleta (County),25111,NaN,2025081101,8,R,2025-08-13T13:42:02.273Z,c4308ef3ad120b43c5e354d74b016b03ba0cbf80,3fd39a94-6af3-455e-9ca6-e827c28e853f
7,7,FM-5526-CO,5526,CO,FM,2024-08-01T00:00:00.000Z,2024,Fire,QUARRY FIRE,False,...,99059,Jefferson (County),24106,NaN,2024080102,8,R,2025-01-08T20:01:25.309Z,c6eff9d2ab33acbeeab7424deae19f7de7e3288d,0d51ec82-33ec-4cf2-91ed-f689507dd98c
8,8,FM-5525-CO,5525,CO,FM,2024-07-31T00:00:00.000Z,2024,Fire,STONE MOUNTAIN FIRE,False,...,99013,Boulder (County),24105,NaN,2024073103,8,R,2025-01-08T20:01:25.309Z,692612f9b487af921c69600ef70b07fd4f95d815,212d16ff-852c-4208-9db5-ef2705e13e37
9,9,FM-5525-CO,5525,CO,FM,2024-07-31T00:00:00.000Z,2024,Fire,STONE MOUNTAIN FIRE,False,...,99069,Larimer (County),24105,NaN,2024073103,8,R,2025-01-08T20:01:25.309Z,b1ce0b36270cc6978ae4fad36df17912f5c1906a,94386e02-7bcc-4451-8999-b0af0728bd39
